# Fase 1 - NDRE Sentinel-2 (2025-2026)
**Auto-detecta** si se ejecuta en Google Colab o en PC local.

- En **Colab**: monta Drive y usa rutas de Drive
- En **Local**: usa rutas del sistema de archivos local

**ROI**: Shapefile (Roigeneral.zip)
**Indice**: NDRE usando bandas NIR (B8) y RedEdge1 (B5)
**Filtro nubes**: cloud_cover < 10% + mascara SCL (clase 4,5)
**Periodo**: 2025-01-01 a 2026-06-03

**NOTA**: Cloud Score+ (CS+) no disponible en STAC (es de GEE).
Se usa SCL + eo:cloud_cover como alternativa de filtrado.


In [ ]:
# CELDA 1: AUTO-DETECCION DE ENTORNO
import sys, subprocess, os, importlib
EN_COLAB = "google.colab" in sys.modules
LIBRERIAS = ['pystac_client', 'geopandas', 'rioxarray', 'rasterio',
            'odc', 'stackstac', 'xarray', 'matplotlib']
LIBRERIAS_FALTANTES = [l for l in LIBRERIAS if not importlib.util.find_spec(l.split('.')[0])]
if EN_COLAB:
    print("Entorno: GOOGLE COLAB")
    if LIBRERIAS_FALTANTES:
        get_ipython().system("pip install pystac-client stackstac rioxarray geopandas rasterio odc-stac -q")
else:
    print("Entorno: PC LOCAL")
    if LIBRERIAS_FALTANTES:
        print(f"Faltan: {LIBRERIAS_FALTANTES}")
        subprocess.check_call([sys.executable, "-m", "pip", "install"] + LIBRERIAS_FALTANTES + ["-q"])
    else:
        print("Todas las librerias ya instaladas.")
print("Entorno listo.")


In [ ]:
# CELDA 2: IMPORTACIONES
import os, glob, calendar, numpy as np
import pandas as pd, geopandas as gpd
import xarray as xr, rioxarray, rasterio
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pystac_client import Client
from odc.stac import load
import unittest
if EN_COLAB:
    from google.colab import drive
print("Importaciones completadas.")


In [ ]:
# =============================================================================
# CELDA 3: CONFIGURACION UNIFICADA
# =============================================================================

if EN_COLAB:
    drive.mount("/content/drive")
    SHAPEFILE_PATH = "/content/drive/MyDrive/Tesis/GEE Murcott/ROI/Roigeneral.zip"
    BASE_DIR = os.path.join(os.path.dirname(SHAPEFILE_PATH), 'ndre2025_2026')
else:
    SCRIPT_DIR = os.getcwd()
    SHAPEFILE_PATH = os.path.join(SCRIPT_DIR, 'Roigeneral.zip')
    if not os.path.exists(SHAPEFILE_PATH):
        SHAPEFILE_PATH = './Roigeneral.zip'
    if not os.path.exists(SHAPEFILE_PATH):
        SHAPEFILE_PATH = input('Ruta del shapefile: ').strip()
    BASE_DIR = os.path.join(SCRIPT_DIR, 'ndre2025_2026')

# Parametros de busqueda
PERIODO_INICIO = "2025-01-01"
PERIODO_FIN = "2026-06-03"
CLOUD_FILTER = {'eo:cloud_cover': {'lt': 10}}

# Nombres de meses en espanol
MES_NOMBRE = ['', 'Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio',
               'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

# Generar lista de fechas por mes
def generar_fechas_por_mes(inicio_str, fin_str):
    inicio = datetime.strptime(inicio_str, '%Y-%m-%d')
    fin = datetime.strptime(fin_str, '%Y-%m-%d')
    fechas = []
    corriente = inicio.replace(day=1)
    while corriente <= fin:
        anio = corriente.year
        mes = corriente.month
        ultimo_dia = calendar.monthrange(anio, mes)[1]
        desde = corriente.strftime('%Y-%m-%d')
        hasta_fin_mes = f'{anio}-{mes:02d}-{ultimo_dia}'
        fecha_hasta_dt = datetime.strptime(hasta_fin_mes, '%Y-%m-%d')
        fecha_hasta = min(fecha_hasta_dt, fin).strftime('%Y-%m-%d')
        label = f'{mes:02d} - {MES_NOMBRE[mes]} {anio}'
        fechas.append((desde, fecha_hasta, label))
        if mes == 12:
            corriente = corriente.replace(year=anio+1, month=1)
        else:
            corriente = corriente.replace(month=mes+1)
    return fechas

FECHAS = generar_fechas_por_mes(PERIODO_INICIO, PERIODO_FIN)

print("Configuracion cargada.")
print(f"  Periodo: {PERIODO_INICIO} a {PERIODO_FIN}")
print(f"  Meses a procesar: {len(FECHAS)}")
print(f"  Filtrar: cloud_cover < 10% + SCL (clase 4,5)")
print(f"  Exportar a: {BASE_DIR}")
for d, h, l in FECHAS:
    print(f'    {l}: {d} -> {h}')


In [ ]:
# =============================================================================
# CELDA 4: FUNCIONES AUXILIARES
# =============================================================================

def calcular_ndre(nir, rededge1):
    """Calcula NDRE = (nir - rededge1) / (nir + rededge1)"""
    ndre = (nir - rededge1) / (nir + rededge1)
    ndre = xr.where((nir + rededge1) == 0, np.nan, ndre)
    return ndre

def aplicar_mascara_scl(scl, clases_validas=[4,5]):
    """Retorna True donde SCL es clase valida"""
    mask = xr.zeros_like(scl, dtype=bool)
    for c in clases_validas:
        mask = mask | (scl == c)
    return mask

def obtener_mes_label(fecha_dt):
    """MM - NombreMes AAAA desde datetime"""
    MES_NOMBRE = ['', 'Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio',
                  'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
    return f"{fecha_dt.month:02d} - {MES_NOMBRE[fecha_dt.month]} {fecha_dt.year}"

def exportar_png(data, path, vmin=0.1, vmax=0.6, dpi=200):
    """Exporta array 2D a PNG con paleta vegetacion GEE"""
    colores = ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']
    cmap = LinearSegmentedColormap.from_list('vigor', colores, N=256)
    cmap.set_bad(color='white', alpha=0)
    plt.figure(figsize=(10,10))
    plt.imshow(np.where(np.isnan(data), np.nan, data), cmap=cmap, vmin=vmin, vmax=vmax)
    plt.axis('off')
    plt.savefig(path, bbox_inches='tight', pad_inches=0, dpi=dpi)
    plt.close()

def generar_csv_metadatos(registros, path):
    """Exporta CSV con metadatos de imagenes procesadas"""
    df = pd.DataFrame(registros)
    columnas = ['fecha', 'hora', 'id_escena', 'nubes_porciento', 'estado_nubosidad', 'ruta_tif', 'ruta_png', 'mes_label']
    df = df[[c for c in columnas if c in df.columns]]
    df.to_csv(path, index=False, encoding='utf-8')
    print(f"CSV exportado: {path} ({len(df)} registros)")

print("Funciones auxiliares cargadas.")
def exportar_tif(da, path, crs='EPSG:4326', nodata=-9999):
    """Exporta xarray DataArray a GeoTIFF con rioxarray"""
    import rasterio, rioxarray
    # Asegurar que rio este configurado
    try:
        da2 = da.copy()
        # Detectar nombres de coordenadas espaciales
        x_name = next((n for n in ['x', 'lon', 'longitude'] if n in da2.coords), None)
        y_name = next((n for n in ['y', 'lat', 'latitude'] if n in da2.coords), None)
        if x_name and y_name:
            da2 = da2.rename({x_name: 'x', y_name: 'y'})
        da2.rio.set_spatial_dims('x', 'y', inplace=True)
        if not da2.rio.crs:
            da2 = da2.rio.write_crs(crs)
        da2.rio.to_raster(path, dtype='float32', compress='lzw', nodata=nodata)
    except Exception as e:
        # Fallback: exportar sin georreferencia
        data = da.values.astype('float32') if hasattr(da, 'values') else da
        ny, nx = data.shape
        with rasterio.open(path, 'w', driver='GTiff', height=ny, width=nx,
            count=1, dtype='float32', crs=rasterio.crs.CRS.from_string(crs),
            compress='lzw', nodata=nodata) as dst:
            dst.write(data, 1)

def aplicar_mascara_geometrica(da, gdf):
    """Mascara: solo pixeles DENTRO de las geometrias del shapefile"""
    from rasterio.features import geometry_mask
    # Obtener coordenadas espaciales
    x_name = next((n for n in ['x', 'lon', 'longitude'] if n in da.coords), None)
    y_name = next((n for n in ['y', 'lat', 'latitude'] if n in da.coords), None)
    if not x_name or not y_name:
        print("  AVISO: No se encontraron coordenadas, saltando mascara geografica")
        return da
    transform = da.rio.transform()
    # out_shape usa el orden (y, x) de las dimensiones
    ny = da.sizes[y_name]
    nx = da.sizes[x_name]
    # geometry_mask retorna True DONDE ESTA LA GEOMETRIA (invertir con ~)
    mascara = ~geometry_mask(gdf.geometry.values, transform=transform, out_shape=(ny, nx))
    # Asegurar que la mascara tenga las dims correctas
    import xarray as xr_m
    mascara_da = xr_m.DataArray(mascara, dims=(y_name, x_name), coords={y_name: da[y_name], x_name: da[x_name]})
    return da.where(mascara_da)


In [ ]:
# =============================================================================
# CELDA 5: CARGAR SHAPEFILE Y CALCULAR BOUNDING BOX
# =============================================================================
print("Cargando shapefile...")
gdf = gpd.read_file(SHAPEFILE_PATH)
print(f"Shapefile cargado: {len(gdf)} feature(s)")
if gdf.crs and gdf.crs.is_geographic:
    gdf_geo = gdf
else:
    gdf_geo = gdf.to_crs('EPSG:4326')
bbox = gdf_geo.total_bounds
print(f"Bounding Box: {bbox}")

# Crear directorio base de exportacion
os.makedirs(BASE_DIR, exist_ok=True)
print(f"Directorio creado: {BASE_DIR}")


In [ ]:
# =============================================================================
# CELDA 6: BUSQUEDA STAC + CONTEO CON FILTRO DE NUBES
# =============================================================================
print("Conectando al catalogo Earth Search (AWS)...")
catalog = Client.open('https://earth-search.aws.element84.com/v1')
print("Conexion exitosa.")

resultados_totales = []
for fecha_inicio, fecha_fin, label in FECHAS:
    print(f"\n{label} ({fecha_inicio} -> {fecha_fin}):")
    search = catalog.search(
        collections=['sentinel-2-l2a'],
        bbox=list(bbox),
        datetime=f'{fecha_inicio}/{fecha_fin}',
        query=CLOUD_FILTER
    )
    items = list(search.items())
    print(f"   Imagenes encontradas: {len(items)}")
    for item in items:
        cloud = item.properties.get('eo:cloud_cover', -1)
        resultados_totales.append({
            'fecha': item.datetime.strftime('%Y-%m-%d') if item.datetime else 'N/A',
            'hora': item.datetime.strftime('%H:%M:%S') if item.datetime else 'N/A',
            'id': item.id,
            'nubes_%': cloud,
            'estado': 'Despejada',
            'mes_label': label,
        })

df_resultados = pd.DataFrame(resultados_totales)
if len(df_resultados) == 0:
    print('\nNo se encontraron imagenes con <10% de nubes.')
    print('Prueba con un filtro menos restrictivo.')
else:
    print(f'\nTotal: {len(df_resultados)} escenas con <10% nubes.')
    print('\nResumen por fecha:')
    for fecha, grupo in df_resultados.groupby('fecha'):
        print(f'  {fecha}: {len(grupo)} img(s) | {grupo["nubes_%"].mean():.1f}% nubes')


In [ ]:
# =============================================================================
# PRUEBAS UNITARIAS
# =============================================================================
class TestFuncionesNDRE(unittest.TestCase):
    def test_calcular_ndre(self):
        nir = xr.DataArray(np.array([0.5]))
        red = xr.DataArray(np.array([0.2]))
        result = calcular_ndre(nir, red).values[0]
        self.assertAlmostEqual(result, 0.4286, places=3)
        # Division por cero
        nir2 = xr.DataArray(np.array([0.0]))
        red2 = xr.DataArray(np.array([0.0]))
        result2 = calcular_ndre(nir2, red2).values[0]
        self.assertTrue(np.isnan(result2))

    def test_aplicar_mascara_scl(self):
        scl = xr.DataArray(np.array([[4, 5, 1, 3, 4, 5, 7, 11]]))
        mask = aplicar_mascara_scl(scl)
        esperado = np.array([[True, True, False, False, True, True, False, False]])
        np.testing.assert_array_equal(mask.values, esperado)

    def test_obtener_mes_label(self):
        from datetime import datetime
        self.assertEqual(obtener_mes_label(datetime(2025, 1, 15)), '01 - Enero 2025')
        self.assertEqual(obtener_mes_label(datetime(2025, 12, 1)), '12 - Diciembre 2025')

suite = unittest.TestLoader().loadTestsFromTestCase(TestFuncionesNDRE)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f'\nTests: {result.testsRun} ejecutados, {len(result.errors)} errores, {len(result.failures)} fallos')


In [ ]:
# =============================================================================
# CELDA 8: PROCESAR NDRE + EXPORTAR POR MES (UN MES A LA VEZ)
# =============================================================================
# NOTA: Procesa un mes, exporta, libera memoria, y sigue al siguiente.
# Si se interrumpe, solo se pierde el mes actual.

registros_csv = []
total_exportados = 0
if len(df_resultados) == 0:
    print("No hay imagenes para cargar. Revisa el filtro de nubes.")
else:
    print(f"Procesando {len(df_resultados)} escenas en {len(FECHAS)} meses...")
    print("Cada mes se procesa y exporta por separado.\n")


    for mes_idx, (fecha_inicio, fecha_fin, label) in enumerate(FECHAS):
        print()
        print("=" * 50)
        print(f'  MES {mes_idx+1}/{len(FECHAS)}: {label}')
        print(f'  {fecha_inicio} -> {fecha_fin}')
        print("=" * 50)

        # Buscar escenas para este mes
        search = catalog.search(
            collections=['sentinel-2-l2a'],
            bbox=list(bbox),
            datetime=f'{fecha_inicio}/{fecha_fin}',
            query=CLOUD_FILTER
        )
        items_mes = list(search.items())
        if len(items_mes) == 0:
            print('  Sin imagenes este mes. Siguiente.')
            continue
        print(f'  Escenas: {len(items_mes)}')

        # Cargar data cube para este mes (en lotes de 5)
        BATCH_SIZE = 5
        ndre_mes = []
        for batch_start in range(0, len(items_mes), BATCH_SIZE):
            batch = items_mes[batch_start:batch_start+BATCH_SIZE]
            ds = load(
                batch,
                bands=['nir', 'rededge1', 'scl'],
                bbox=list(bbox),
                crs='EPSG:4326',
                resolution=0.0001,
                groupby=None,
            )
            if ds.sizes.get('time', 0) == 0:
                continue

            for t in range(ds.sizes['time']):
                escena = ds.isel(time=t)
                nir = escena['nir']
                rededge1 = escena['rededge1']
                scl = escena['scl']
                ndre = calcular_ndre(nir, rededge1)
                mascara = aplicar_mascara_scl(scl)
                ndre_masked = ndre.where(mascara)
                # Recortar a las parcelas del shapefile
                ndre_masked = aplicar_mascara_geometrica(ndre_masked, gdf_geo)
                ts_val = ds.time.values[t]
                fecha_dt = pd.Timestamp(ts_val).to_pydatetime()
                ndre_mes.append({
                    'ndre': ndre_masked,
                    'fecha_dt': fecha_dt,
                })
            del ds

        print(f'  NDRE calculado: {len(ndre_mes)} imagenes')

        # Exportar TIF + PNG para este mes
        contador = 0  # Reiniciar contador por mes
        registros_mes = []  # Lista separada para este mes
        mes_carpeta = os.path.join(BASE_DIR, label)
        tif_dir = os.path.join(mes_carpeta, 'TIF')
        png_dir = os.path.join(mes_carpeta, 'PNG')
        os.makedirs(tif_dir, exist_ok=True)
        os.makedirs(png_dir, exist_ok=True)

        for resultado in ndre_mes:
            ndre_data = resultado['ndre']
            fecha_dt = resultado['fecha_dt']
            data_values = ndre_data.values if hasattr(ndre_data, 'values') else ndre_data
            if np.all(np.isnan(data_values)):
                continue

            fecha_str = fecha_dt.strftime('%Y%m%d')
            hora_str = fecha_dt.strftime('%H%M%S')
            # Nombre incremental por mes
            contador += 1
            # Extraer nombre del mes en espanol de label
            mes_corto = label.split(' - ')[1].split()[0] if ' - ' in label else label
            nombre_base = f'NDRE_{mes_corto}_{contador:03d}_{fecha_str}_{hora_str}'
            tif_path = os.path.join(tif_dir, f'{nombre_base}.tif')
            png_path = os.path.join(png_dir, f'{nombre_base}.png')

            exportar_tif(ndre_data, tif_path, crs='EPSG:4326')
            exportar_png(data_values, png_path)

            total_exportados += 1
            registros_mes.append({
                'fecha': fecha_dt.strftime('%Y-%m-%d'),
                'hora': fecha_dt.strftime('%H:%M:%S'),
                'id_escena': nombre_base,
                'nubes_porciento': '',
                'estado_nubosidad': 'Filtrada (<10%)',
                'ruta_tif': tif_path,
                'ruta_png': png_path,
                'mes_label': label,
            })

        print(f'  Exportado: {len(ndre_mes)} imagenes a {label}')
        # CSV de metadatos para este mes
        if len(registros_mes) > 0:
            csv_path = os.path.join(mes_carpeta, 'metadatos.csv')
            generar_csv_metadatos(registros_mes, csv_path)
        # Liberar memoria del mes antes del siguiente
        del ndre_mes

    print(f'\nProcesamiento completo: {total_exportados} imagenes en total.')


In [ ]:
# ============================================================
# CELDA 9: REPORTE FINAL
# ============================================================

print('=' * 50)
print('   REPORTE FINAL - NDRE 2025-2026')
print('=' * 50)
print(f'  Total imagenes exportadas: {total_exportados}')
print(f'  Directorio base: {BASE_DIR}')
print()
for carpeta_mes in sorted(os.listdir(BASE_DIR)):
    tif_path_mes = os.path.join(BASE_DIR, carpeta_mes, 'TIF')
    png_path_mes = os.path.join(BASE_DIR, carpeta_mes, 'PNG')
    csv_path_mes = os.path.join(BASE_DIR, carpeta_mes, 'metadatos.csv')
    if os.path.isdir(tif_path_mes):
        tif_c = len(glob.glob(os.path.join(tif_path_mes, '*.tif')))
        png_c = len(glob.glob(os.path.join(png_path_mes, '*.png')))
        csv_ok = 'CSV' if os.path.exists(csv_path_mes) else ''
        if tif_c > 0 or png_c > 0:
            print(f'  {carpeta_mes}: {tif_c} TIFs, {png_c} PNGs {csv_ok}')
print('=' * 50)
print("  NDRE download complete!")
print('=' * 50)
